# 🔎 Curiosidade — Sobral vs. Municípios Destaque | IEGM 2023

## Contexto

Durante a análise principal, foi possível observar que **Sobral** liderou em cinco das sete áreas do IEGM 2023. No entanto, nas áreas de **Educação** e **Planejamento**, outros municípios obtiveram notas superiores — **Groaíras** e **Caucaia**, respectivamente.

Esse notebook nasce de uma curiosidade: **o que esses municípios fizeram de diferente?**

## O que será investigado

A base de respostas do IEGM funciona como um questionário estruturado enviado por cada prefeitura ao Tribunal de Contas. Cada resposta possui uma chave, uma descrição e uma nota atribuída — e a forma como essas respostas são organizadas segue um padrão hierárquico próprio.

O objetivo aqui é ir além da comparação de notas e entender:

- **Como as respostas foram estruturadas e enviadas** por cada município
- **Qual o padrão de preenchimento** do questionário — a ordem, a hierarquia e a relação entre as respostas
- **O que Groaíras e Caucaia responderam de diferente em relação a Sobral** nas áreas onde ficaram à frente
- **Se existe algum padrão nas respostas que pontuaram** que possa explicar a diferença de desempenho

> 📁 Este notebook é um desdobramento da análise principal `analise_iegm_2023_ceara.ipynb` e utiliza as mesmas bases de dados do TCE-CE, ano de referência 2023.

In [1]:
# ============================================================
# Importação das bibliotecas necessárias
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## Carregamento dos Dados

In [2]:
# ============================================================
# Carregamento das bases de dados
# ============================================================

# Base 1: Notas gerais por município
df_geral = pd.read_csv('geral_iegm_2023_TCECE_municipio.csv', sep=';', decimal=',', encoding='utf-16')

# Base 2: Respostas individuais — leitura manual para preservar campos com ';' no texto
linhas = []
header = None

with open('respostas_iegm_2023_TCECE_completo_nota.csv', 'r', encoding='utf-16') as f:
    for i, linha in enumerate(f):
        campos = linha.rstrip('\n').split(';')
        if i == 0:
            header = campos
            n_campos = len(campos)
        else:
            if len(campos) > n_campos:
                campos = campos[:n_campos-1] + [';'.join(campos[n_campos-1:])]
            if len(campos) == n_campos:
                linhas.append(campos)

df_respostas = pd.DataFrame(linhas, columns=header)
df_respostas['nota'] = pd.to_numeric(df_respostas['nota'].str.replace(',', '.'), errors='coerce')

# Base 3: Cálculo detalhado das notas por área
df_calculo = pd.read_csv('calculo_iegm_2023_TCECE_completo.csv', sep=';', decimal=',', encoding='utf-16')

print("Bases carregadas com sucesso:")
print(f"\n df_geral:     {df_geral.shape[0]} linhas | {df_geral.shape[1]} colunas")
print(f" df_respostas: {df_respostas.shape[0]} linhas | {df_respostas.shape[1]} colunas")
print(f" df_calculo:   {df_calculo.shape[0]} linhas | {df_calculo.shape[1]} colunas")

Bases carregadas com sucesso:

 df_geral:     182 linhas | 22 colunas
 df_respostas: 199285 linhas | 18 colunas
 df_calculo:   1274 linhas | 15 colunas


In [3]:
# =====================================================
# MELHORES MUNICÍPIOS POR ÁREA — IEGM 2023
# =====================================================

areas = {
    'pct_iamb':     '🌿 Meio Ambiente',
    'pct_icidade':  '🏙️ Proteção das Cidades',
    'pct_ieduc':    '📚 Educação',
    'pct_ifiscal':  '💰 Gestão Fiscal',
    'pct_igov_ti':  '💻 Governança de TI',
    'pct_isaude':   '🏥 Saúde',
    'pct_iplan':    '📋 Planejamento',
}

resultados = []

for coluna, nome_area in areas.items():
    idx_max = df_geral[coluna].idxmax()
    linha = df_geral.loc[idx_max]
    col_ind = coluna.replace('pct_', 'ind_')
    faixa = linha[col_ind] if col_ind in df_geral.columns else '-'

    resultados.append({
        'Área': nome_area,
        'Município': linha['municipio'].title(),
        'Nota (%)': round(linha[coluna] * 100, 2) if linha[coluna] <= 1 else round(linha[coluna], 2),
        'Faixa': faixa
    })

df_melhores = pd.DataFrame(resultados)

print("=" * 60)
print("MELHORES MUNICÍPIOS POR ÁREA — IEGM 2023 | CEARÁ")
print("=" * 60)
display(df_melhores)

MELHORES MUNICÍPIOS POR ÁREA — IEGM 2023 | CEARÁ


,Área,Município,Nota (%),Faixa
0,🌿 Meio Ambiente,Sobral,100.0,A
1,🏙️ Proteção das Cidades,Sobral,100.0,A
2,📚 Educação,Groaíras,56.0,C+
3,💰 Gestão Fiscal,Sobral,98.0,A
4,💻 Governança de TI,Sobral,100.0,A
5,🏥 Saúde,Sobral,93.0,A
6,📋 Planejamento,Caucaia,90.0,A


In [4]:
# =====================================================
# MAPA INDICADOR
# =====================================================

mapa_indicador = {
    'pct_iamb':    'i-Amb',
    'pct_icidade': 'i-Cidade',
    'pct_ieduc':   'i-Educ',
    'pct_ifiscal': 'i-Fiscal',
    'pct_igov_ti': 'i-Gov TI',
    'pct_isaude':  'i-Saude',
    'pct_iplan':   'i-Plan',
}

def get_respostas(coluna_pct):
    # Busca direto pelo índice do df_melhores
    idx = list(mapa_indicador.keys()).index(coluna_pct)
    row = df_melhores.iloc[idx]

    municipio = row['Município'].upper()
    indicador = mapa_indicador[coluna_pct]

    print(f"Buscando: municipio='{municipio}' | indicador='{indicador}'")

    df = df_respostas[
        (df_respostas['municipio'] == municipio) &
        (df_respostas['indicador'] == indicador)
    ][['chave_questao', 'questao', 'chave_resposta', 'resposta', 'nota']].copy()

    df.columns = ['Chave Questão', 'Questão', 'Chave Resposta', 'Resposta', 'Nota']
    return df.reset_index(drop=True), row

### COMPARAÇÃO I-EDUC — Groaíras vs Sobral

In [5]:
# Indicador analisado
indicador = 'i-Educ'

row_educ = df_melhores.iloc[list(mapa_indicador.keys()).index('pct_ieduc')]
municipio_camp = row_educ['Município'].upper()

cols = ['indice_questao', 'chave_questao', 'questao', 'chave_resposta', 'resposta', 'nota']

df_camp = df_respostas[
    (df_respostas['municipio'] == municipio_camp) &
    (df_respostas['indicador'] == indicador)
][cols].reset_index(drop=True)

df_sobral = df_respostas[
    (df_respostas['municipio'] == 'SOBRAL') &
    (df_respostas['indicador'] == indicador)
][cols].reset_index(drop=True)

# Monta lado a lado por posição — sem merge
df_comp_educ = pd.DataFrame({
    'Índice Questão':                    df_camp['indice_questao'],
    'Chave Questão':                     df_camp['chave_questao'],
    'Questão':                           df_camp['questao'],
    f'Chave Resposta {municipio_camp}':  df_camp['chave_resposta'],
    f'Resposta {municipio_camp}':        df_camp['resposta'],
    f'Nota {municipio_camp}':            df_camp['nota'],
    f'Chave Resposta SOBRAL':            df_sobral['chave_resposta'],
    f'Resposta SOBRAL':                  df_sobral['resposta'],
    f'Nota SOBRAL':                      df_sobral['nota'],
})

print(f"EDUCAÇÃO — {municipio_camp} ({row_educ['Nota (%)']}% [{row_educ['Faixa']}]) vs SOBRAL")
print(f"Linhas {municipio_camp}: {len(df_camp)} | Linhas SOBRAL: {len(df_sobral)}")
display(df_comp_educ)

df_comp_educ.to_csv(f'comparacao_ieduc_{municipio_camp}_vs_SOBRAL.csv', index=False, sep=';', encoding='utf-8-sig')
print(f"CSV salvo: comparacao_ieduc_{municipio_camp}_vs_SOBRAL.csv")

EDUCAÇÃO — GROAÍRAS (56.0% [C+]) vs SOBRAL
Linhas GROAÍRAS: 418 | Linhas SOBRAL: 450


,Índice Questão,Chave Questão,Questão,Chave Resposta GROAÍRAS,Resposta GROAÍRAS,Nota GROAÍRAS,Chave Resposta SOBRAL,Resposta SOBRAL,Nota SOBRAL
0,001.,M21Q06420R00200,Informe quantos estabelecimentos que oferecem Creche possuem Pátio Infantil (PI):,,1,0.666667,,29,1.348837
1,002.,M01Q06431R00100,Informe quantos estabelecimentos que oferecem creche disponibilizam brinquedos/materiais pedagógicos:,,3,4.000000,,37,3.441860
2,003.,M01Q06492R00100,Informe o número de crianças matriculadas na creche:,,299,0.000000,,4943,0.000000
3,004.,M01Q06495R00100,Informe a data de início de funcionamento para as creches:,,31/01/2022,0.000000,,01/02/2022,0.000000
4,005.,M01Q06480R00100,Informe a quantidade de estabelecimentos que oferecem Creche na rede municipal de ensino:,,3,0.000000,,43,0.000000
...,...,...,...,...,...,...,...,...,...
445,NaN,NaN,NaN,NaN,NaN,NaN,,"Aos Professores de distritos são ofertados serviços de transporte para deslocamento até o local da formação e retorno à sua localidade. Formação realizada dentro do turno de trabalho, contemplando o tempo destinado para planejamento, contemplado pelo ? de carga horária destinada para estudos.",0.000000
446,NaN,NaN,NaN,NaN,NaN,NaN,,https://drive.google.com/drive/folders/1DElTc21md1ngUTU_8a7vua4TrGoDd_Pz,0.000000
447,NaN,NaN,NaN,NaN,NaN,NaN,D01Q01100R00500,Formação inicial docente deficiente para a educação das relações étnico-raciais.,0.000000
448,NaN,NaN,NaN,NaN,NaN,NaN,M01Q06300R00100,Sim,0.000000


CSV salvo: comparacao_ieduc_GROAÍRAS_vs_SOBRAL.csv


### COMPARAÇÃO I-PLAN — Caucaia vs Sobral

In [6]:
# Indicador analisado
indicador = 'i-Plan'

row_plan = df_melhores.iloc[list(mapa_indicador.keys()).index('pct_iplan')]
municipio_camp = row_plan['Município'].upper()

cols = ['indice_questao', 'chave_questao', 'questao', 'chave_resposta', 'resposta', 'nota']

df_camp = df_respostas[
    (df_respostas['municipio'] == municipio_camp) &
    (df_respostas['indicador'] == indicador)
][cols].reset_index(drop=True)

df_sobral = df_respostas[
    (df_respostas['municipio'] == 'SOBRAL') &
    (df_respostas['indicador'] == indicador)
][cols].reset_index(drop=True)

# Monta lado a lado por posição — sem merge
df_comp_plan = pd.DataFrame({
    'Índice Questão':                  df_camp['indice_questao'],
    'Chave Questão':                   df_camp['chave_questao'],
    'Questão':                         df_camp['questao'],
    f'Chave Resposta {municipio_camp}': df_camp['chave_resposta'],
    f'Resposta {municipio_camp}':       df_camp['resposta'],
    f'Nota {municipio_camp}':           df_camp['nota'],
    f'Chave Resposta SOBRAL':           df_sobral['chave_resposta'],
    f'Resposta SOBRAL':                 df_sobral['resposta'],
    f'Nota SOBRAL':                     df_sobral['nota'],
})

print(f"PLANEJAMENTO — {municipio_camp} ({row_plan['Nota (%)']}% [{row_plan['Faixa']}]) vs SOBRAL")
print(f"Linhas {municipio_camp}: {len(df_camp)} | Linhas SOBRAL: {len(df_sobral)}")
display(df_comp_plan)

df_comp_plan.to_csv(f'comparacao_iplan_{municipio_camp}_vs_SOBRAL.csv', index=False, sep=';', encoding='utf-8-sig')
print(f"CSV salvo: comparacao_iplan_{municipio_camp}_vs_SOBRAL.csv")

PLANEJAMENTO — CAUCAIA (90.0% [A]) vs SOBRAL
Linhas CAUCAIA: 478 | Linhas SOBRAL: 357


,Índice Questão,Chave Questão,Questão,Chave Resposta CAUCAIA,Resposta CAUCAIA,Nota CAUCAIA,Chave Resposta SOBRAL,Resposta SOBRAL,Nota SOBRAL
0,001.,M03Q08000,"O planejamento da prefeitura, para o ano de 2022 foi estruturado através de programas, indicadores, metas e ações",M03Q08000R00100,Sim,0.0,M03Q08000R00100,Sim,0.0
1,001.001.,M03Q08100R00100,Quantidade de Programas:,,24,500.0,,53,0.0
2,001.001.001.,CODPROG,Código do Programa,,101,0.0,,101,0.0
3,001.001.002.,DESPROG,Descrição do Programa,,GEST?O DA ARTE E DA CULTURA,0.0,,MODERNIZAÇÃO DA GESTÃO PÚBLICA E RECURSOS LOGÍSTICOS,0.0
4,001.001.005.,VLRESTI,Valor Estimado do Indicador,,"12,0000",0.0,,"100,0000",0.0
...,...,...,...,...,...,...,...,...,...
473,021.,M03Q08200R00100,Informe o valor total da dotação inicial autorizada pela Lei Orçamentária Anual (LOA) para o ano de 2023,,"1548044000,0000",0.0,NaN,NaN,NaN
474,022.,M03Q08200R00200,Informe o valor total da dotação atualizada em 31/12/2023,,"1548044000,0000",0.0,NaN,NaN,NaN
475,023.,M03Q08400,Pontualidade na Entrega dos Documentos relativos às Peças de Planejamento,M03Q08400R00100,Planejamento entregue no prazo,150.0,NaN,NaN,NaN
476,024.,M03Q08500,O município aderiu ao Programa Nacional de Prevenção à Corrupção (PNPC) por meio do questionário de integridade no sistema e-Prevenção?,M03Q08500R00100,Sim,0.0,NaN,NaN,NaN


CSV salvo: comparacao_iplan_CAUCAIA_vs_SOBRAL.csv
